In [ ]:
import glob
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import root_mean_squared_error, mean_squared_error

In [ ]:
FIELD_X_MIN, FIELD_X_MAX = 0.0, 120.0
FIELD_Y_MIN, FIELD_Y_MAX = 0.0, 53.3

In [ ]:
input_files = sorted(glob.glob("train/input_2023_w*.csv"))
output_files = sorted(glob.glob("train/output_2023_w*.csv"))

df_input = pd.concat([pd.read_csv(f) for f in input_files], ignore_index=True)
df_output = pd.concat([pd.read_csv(f) for f in output_files], ignore_index=True)

print("Input shape:", df_input.shape)
print("Output shape:", df_output.shape)

# Make sure player_to_predict is int
df_input["player_to_predict"] = df_input["player_to_predict"].astype(int)

# Engineer Features

## Base Features

In [ ]:
def parse_height_to_inches(h):
    if pd.isna(h):
        return np.nan
    try:
        ft, inch = str(h).split("-")
        return int(ft) * 12 + int(inch)
    except Exception:
        return np.nan
    
def get_velocity(speed, direction_deg):
    theta = np.deg2rad(direction_deg)
    return speed * np.sin(theta), speed * np.cos(theta)

In [ ]:
df_input["height_in"] = df_input["player_height"].apply(parse_height_to_inches)

df_input["age"] = 2026 - pd.to_datetime(df_input["player_birth_date"]).dt.year

# Assume x in [0, 120], y in [0, 53.3]
# For plays moving left, flip coordinates.
is_left = df_input["play_direction"].str.lower() == "left"

# simple numeric encoding for play_direction if needed
df_input["play_dir_num"] = np.where(is_left, -1, 1)

In [ ]:
df_input['bmi'] = (df_input['player_weight'] / (df_input['height_in']**2)) * 703

dir_rad = np.deg2rad(df_input['dir'].fillna(0))
df_input['velocity_x'] = df_input['s'] * np.sin(dir_rad)
df_input['velocity_y'] = df_input['s'] * np.cos(dir_rad)
df_input['acceleration_x'] = df_input['a'] * np.cos(dir_rad)
df_input['acceleration_y'] = df_input['a'] * np.sin(dir_rad)

df_input['is_offense'] = (df_input['player_side'] == 'Offense').astype(int)
df_input['is_defense'] = (df_input['player_side'] == 'Defense').astype(int)
df_input['is_receiver'] = (df_input['player_role'] == 'Targeted Receiver').astype(int)
df_input['is_coverage'] = (df_input['player_role'] == 'Defensive Coverage').astype(int)
df_input['is_passer'] = (df_input['player_role'] == 'Passer').astype(int)

df_input['role_targeted_receiver'] = df_input['is_receiver']
df_input['role_defensive_coverage'] = df_input['is_coverage']
df_input['role_passer'] = df_input['is_passer']
df_input['side_offense'] = df_input['is_offense']

df_input['momentum_x'] = df_input['velocity_x'] * df_input['player_weight']
df_input['momentum_y'] = df_input['velocity_y'] * df_input['player_weight']
df_input['kinetic_energy'] = 0.5 * df_input['player_weight'] * (df_input['s'] ** 2)

df_input['speed_squared'] = df_input['s'] ** 2
df_input['accel_magnitude'] = np.sqrt(df_input['acceleration_x']**2 + df_input['acceleration_y']**2)
df_input['orientation_diff'] = np.abs(df_input['o'] - df_input['dir'])
df_input['orientation_diff'] = np.minimum(df_input['orientation_diff'], 360 - df_input['orientation_diff'])

if 'ball_land_x' in df_input.columns:
    ball_dx = df_input['ball_land_x'] - df_input['x']
    ball_dy = df_input['ball_land_y'] - df_input['y']
    df_input['distance_to_ball'] = np.sqrt(ball_dx**2 + ball_dy**2)
    df_input['dist_to_ball'] = df_input['distance_to_ball']
    df_input['dist_squared'] = df_input['distance_to_ball'] ** 2
    df_input['angle_to_ball'] = np.arctan2(ball_dy, ball_dx)
    df_input['ball_direction_x'] = ball_dx / (df_input['distance_to_ball'] + 1e-6)
    df_input['ball_direction_y'] = ball_dy / (df_input['distance_to_ball'] + 1e-6)
    df_input['closing_speed_ball'] = (
        df_input['velocity_x'] * df_input['ball_direction_x'] +
        df_input['velocity_y'] * df_input['ball_direction_y']
    )
    df_input['velocity_toward_ball'] = (
        df_input['velocity_x'] * np.cos(df_input['angle_to_ball']) + 
        df_input['velocity_y'] * np.sin(df_input['angle_to_ball'])
    )
    df_input['velocity_alignment'] = np.cos(df_input['angle_to_ball'] - dir_rad)
    df_input['angle_diff'] = np.abs(df_input['o'] - np.degrees(df_input['angle_to_ball']))
    df_input['angle_diff'] = np.minimum(df_input['angle_diff'], 360 - df_input['angle_diff'])

In [ ]:
# In/near red zone: close to opponent end zone
df_input["is_red_zone"] = (df_input["absolute_yardline_number"] <= 20).astype(int)

# Backed up near own end zone
df_input["is_backed_up"] = (df_input["absolute_yardline_number"] >= 80).astype(int)

## Lag Features

In [ ]:
df_input = df_input.copy()
    
df_input = df_input.sort_values(['game_id', 'play_id', 'nfl_id', 'frame_id'])
gcols = ['game_id', 'play_id', 'nfl_id']

for lag in [1, 2, 3, 4, 5]:
    for col in ['x', 'y', 'velocity_x', 'velocity_y', 's', 'a']:
        if col in df_input.columns:
            df_input[f'{col}_lag{lag}'] = df_input.groupby(gcols)[col].shift(lag)

for window in [3, 5]:
    for col in ['x', 'y', 'velocity_x', 'velocity_y', 's']:
        if col in df_input.columns:
            df_input[f'{col}_rolling_mean_{window}'] = (
                df_input.groupby(gcols)[col]
                .rolling(window, min_periods=1).mean()
                .reset_index(level=[0,1,2], drop=True)
            )
            df_input[f'{col}_rolling_std_{window}'] = (
                df_input.groupby(gcols)[col]
                .rolling(window, min_periods=1).std()
                .reset_index(level=[0,1,2], drop=True)
            )

for col in ['velocity_x', 'velocity_y']:
    if col in df_input.columns:
        df_input[f'{col}_delta'] = df_input.groupby(gcols)[col].diff()

df_input['velocity_x_ema'] = df_input.groupby(gcols)['velocity_x'].transform(
    lambda x: x.ewm(alpha=0.3, adjust=False).mean()
)
df_input['velocity_y_ema'] = df_input.groupby(gcols)['velocity_y'].transform(
    lambda x: x.ewm(alpha=0.3, adjust=False).mean()
)
df_input['speed_ema'] = df_input.groupby(gcols)['s'].transform(
    lambda x: x.ewm(alpha=0.3, adjust=False).mean()
)

## Opponent Features

In [ ]:
features = []
    
for (gid, pid), group in df_input.groupby(['game_id', 'play_id']):
    last = group.sort_values('frame_id').groupby('nfl_id').last()

    if len(last) < 2:
        continue

    positions = last[['x', 'y']].values
    sides = last['player_side'].values
    speeds = last['s'].values
    directions = last['dir'].values
    roles = last['player_role'].values

    receiver_mask = np.isin(roles, ['Targeted Receiver', 'Other Route Runner'])

    for i, (nid, side, role) in enumerate(zip(last.index, sides, roles)):
        opp_mask = sides != side

        feat = {
            'game_id': gid, 'play_id': pid, 'nfl_id': nid,
            'nearest_opp_dist': 50.0, 'closing_speed': 0.0,
            'num_nearby_opp_3': 0, 'num_nearby_opp_5': 0,
            'mirror_wr_vx': 0.0, 'mirror_wr_vy': 0.0,
            'mirror_offset_x': 0.0, 'mirror_offset_y': 0.0,
            'mirror_wr_dist': 50.0,
        }

        if not opp_mask.any():
            features.append(feat)
            continue

        opp_positions = positions[opp_mask]
        distances = np.sqrt(((positions[i] - opp_positions) ** 2).sum(axis=1))

        if len(distances) == 0:
            features.append(feat)
            continue

        nearest_idx = distances.argmin()
        feat['nearest_opp_dist'] = distances[nearest_idx]
        feat['num_nearby_opp_3'] = (distances < 3.0).sum()
        feat['num_nearby_opp_5'] = (distances < 5.0).sum()

        my_vx, my_vy = get_velocity(speeds[i], directions[i])
        opp_speeds = speeds[opp_mask]
        opp_dirs = directions[opp_mask]
        opp_vx, opp_vy = get_velocity(opp_speeds[nearest_idx], opp_dirs[nearest_idx])

        rel_vx = my_vx - opp_vx
        rel_vy = my_vy - opp_vy
        to_me = positions[i] - opp_positions[nearest_idx]
        to_me_norm = to_me / (np.linalg.norm(to_me) + 0.1)
        feat['closing_speed'] = -(rel_vx * to_me_norm[0] + rel_vy * to_me_norm[1])

        if role == 'Defensive Coverage' and receiver_mask.any():
            rec_positions = positions[receiver_mask]
            rec_distances = np.sqrt(((positions[i] - rec_positions) ** 2).sum(axis=1))

            if len(rec_distances) > 0:
                closest_rec_idx = rec_distances.argmin()
                rec_indices = np.where(receiver_mask)[0]
                actual_rec_idx = rec_indices[closest_rec_idx]

                rec_vx, rec_vy = get_velocity(speeds[actual_rec_idx], directions[actual_rec_idx])

                feat['mirror_wr_vx'] = rec_vx
                feat['mirror_wr_vy'] = rec_vy
                feat['mirror_wr_dist'] = rec_distances[closest_rec_idx]
                feat['mirror_offset_x'] = positions[i][0] - rec_positions[closest_rec_idx][0]
                feat['mirror_offset_y'] = positions[i][1] - rec_positions[closest_rec_idx][1]

        features.append(feat)

# Format Taining Data

In [ ]:
df_input = df_input.merge(pd.DataFrame(features), on=['game_id', 'play_id', 'nfl_id'], how='left')

In [ ]:
print(df_input.columns.to_list())

In [ ]:
id_cols = ["game_id", "play_id", "nfl_id"]

In [ ]:
df_last_in = (
    df_input
    .sort_values("frame_id")
    .groupby(id_cols, as_index=False)
    .tail(1)  # last frame before/at throw for that player
)

In [ ]:
df_last_in = df_last_in[df_last_in["player_to_predict"] == 1].copy()

In [ ]:
cols_for_snapshot = id_cols + [

    # --- Target marker ---
    "player_to_predict",

    # --- Play context ---
    "play_dir_num",            # numeric encoded direction (if you created it)
    "absolute_yardline_number",
    "is_red_zone",
    "is_backed_up",
    "num_frames_output",

    # --- Raw spatial ---
    "x", "y",
    "ball_land_x", "ball_land_y",

    # --- Kinematic raw ---
    "s",                      # speed
    "a",                      # acceleration
    "dir",                    # motion direction
    "o",                      # orientation

    # --- Engineered physical attributes ---
    "player_weight",
    "height_in",
    "age",
    "bmi",

    # --- Velocity / Acceleration components ---
    "velocity_x",
    "velocity_y",
    "acceleration_x",
    "acceleration_y",

    # --- Momentum / Energy ---
    "momentum_x",
    "momentum_y",
    "kinetic_energy",
    "speed_squared",
    "accel_magnitude",

    # --- Angles / Orientation relationships ---
    "orientation_diff",
    "angle_to_ball",
    "angle_diff",

    # --- Distances ---
    "distance_to_ball",
    "dist_to_ball",
    "dist_squared",

    # --- Ball-directional features ---
    "ball_direction_x",
    "ball_direction_y",
    "closing_speed_ball",
    "velocity_toward_ball",
    "velocity_alignment",

    # --- Role + Side flags ---
    "is_offense",
    "is_defense",
    "is_receiver",
    "is_coverage",
    "is_passer",
    "role_targeted_receiver",
    "role_defensive_coverage",
    "role_passer",
    "side_offense",

    # --- Temporal lagged features ---
    "x_lag1", "y_lag1",
    "velocity_x_lag1", "velocity_y_lag1",
    "s_lag1", "a_lag1",

    "x_lag2", "y_lag2",
    "velocity_x_lag2", "velocity_y_lag2",
    "s_lag2", "a_lag2",

    "x_lag3", "y_lag3",
    "velocity_x_lag3", "velocity_y_lag3",
    "s_lag3", "a_lag3",

    "x_lag4", "y_lag4",
    "velocity_x_lag4", "velocity_y_lag4",
    "s_lag4", "a_lag4",

    "x_lag5", "y_lag5",
    "velocity_x_lag5", "velocity_y_lag5",
    "s_lag5", "a_lag5",

    # --- Rolling window stats ---
    "x_rolling_mean_3", "x_rolling_std_3",
    "y_rolling_mean_3", "y_rolling_std_3",
    "velocity_x_rolling_mean_3", "velocity_x_rolling_std_3",
    "velocity_y_rolling_mean_3", "velocity_y_rolling_std_3",
    "s_rolling_mean_3", "s_rolling_std_3",

    "x_rolling_mean_5", "x_rolling_std_5",
    "y_rolling_mean_5", "y_rolling_std_5",
    "velocity_x_rolling_mean_5", "velocity_x_rolling_std_5",
    "velocity_y_rolling_mean_5", "velocity_y_rolling_std_5",
    "s_rolling_mean_5", "s_rolling_std_5",

    # --- Derived velocity deltas / EMAs ---
    "velocity_x_delta",
    "velocity_y_delta",
    "velocity_x_ema",
    "velocity_y_ema",
    "speed_ema",

    # --- Opponent context ---
    "nearest_opp_dist",
    "closing_speed",
    "num_nearby_opp_3",
    "num_nearby_opp_5",

    # --- Mirror coverage features ---
    "mirror_wr_vx",
    "mirror_wr_vy",
    "mirror_offset_x",
    "mirror_offset_y",
    "mirror_wr_dist",
]


In [ ]:
cols_for_snapshot = [c for c in cols_for_snapshot if c in df_last_in.columns]

df_last_in = df_last_in[cols_for_snapshot]

In [ ]:
df_train = df_output.merge(df_last_in, on=id_cols, how="inner", suffixes=('','_out'))
df_train = df_train.reset_index(drop=True)

In [ ]:
df_train["t_norm"] = df_train["frame_id"] / df_train["num_frames_output"]

In [ ]:
df_train = df_train[df_train["player_to_predict"] == 1].copy()

# Drop any rows with missing targets
df_train = df_train.dropna(subset=["x_out", "y_out"])

print("Train rows for player_to_predict=1:", df_train.shape[0])

In [ ]:
target_cols = ["x_out", "y_out"]

# Columns we definitely don't want as features
cols_to_exclude = set(id_cols + [
    "frame_id",         # raw output coords (we use standardized targets)
    "play_direction",   # encoded via standardized coords
    "num_frames_output" # we used it to build t_norm
])

# Also exclude target columns themselves
cols_to_exclude.update(target_cols)

feature_cols = [c for c in df_train.columns if c not in cols_to_exclude]

X = df_train[feature_cols].values
y = df_train[target_cols].values

print(f"Using {len(feature_cols)} feature columns:", feature_cols)

In [ ]:
groups = df_train["game_id"].values  # group by game

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=17)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])

In [ ]:
base_reg = HistGradientBoostingRegressor(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    min_samples_leaf=30,
    random_state=17
)

model = MultiOutputRegressor(base_reg)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

contest_rmse = np.sqrt(0.5 * (mean_squared_error(y_test[:, 0], y_pred[:, 0]) + mean_squared_error(y_test[:, 1], y_pred[:, 1])))

print(f"RMSE: {contest_rmse:.4f}")